# 서울시 노인의료복지시설 주소 VWorld 지오코딩

`Data/서울시_노인의료복지시설_주소목록.csv`의 `기관소재지` 주소를 VWorld Geocoder API로 조회해 `경도`, `위도` 컬럼을 추가합니다.

생성 파일: `Data/서울시_노인의료복지시설현황_geocoded.csv`

In [4]:
from pathlib import Path
import os
import re
import time

import pandas as pd
import requests

INPUT_CSV = Path("Data/서울시_노인의료복지시설_주소목록.csv")
OUTPUT_CSV = Path("Data/서울시_노인의료복지시설현황_geocoded.csv")
FAILED_CSV = Path("Data/서울시_노인의료복지시설현황_geocoding_failed.csv")

VWORLD_GEOCODE_URL = "https://api.vworld.kr/req/address"

# 방법 1: 아래 문자열에 VWorld 인증키를 직접 넣습니다.
# 방법 2: 환경변수 VWORLD_API_KEY에 인증키를 설정해도 됩니다.
VWORLD_API_KEY = "B8F5DE7D-4482-3614-B496-6D8B6E1B1043"

# API 요청 간격입니다. 과도한 연속 요청을 피하기 위해 약간의 대기 시간을 둡니다.
SLEEP_SECONDS = 0.05
REQUEST_TIMEOUT = 10

print(f"입력 CSV: {INPUT_CSV}")
print(f"출력 CSV: {OUTPUT_CSV}")

입력 CSV: Data/서울시_노인의료복지시설_주소목록.csv
출력 CSV: Data/서울시_노인의료복지시설현황_geocoded.csv


In [5]:
def normalize_address_candidates(address):
    """VWorld API 조회 성공률을 높이기 위해 원본 주소와 정리된 주소 후보를 만듭니다."""
    address = str(address).strip()
    if not address or address.lower() == "nan":
        return []

    candidates = [address]

    # 괄호 안 상세 설명, 중복 공백 등을 제거합니다.
    cleaned = re.sub(r"\([^)]*\)", " ", address)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # 원본 데이터는 '서울시'로 시작하는 주소가 많으므로 VWorld 검색에 안정적인 '서울특별시'로 바꿉니다.
    cleaned = cleaned.replace("서울시 ", "서울특별시 ")
    cleaned = re.sub(r"^서울\s+", "서울특별시 ", cleaned)

    # 콤마 뒤 상세주소가 있는 경우 도로명주소 본문만 별도 후보로 사용합니다.
    if "," in cleaned:
        candidates.append(cleaned.split(",", maxsplit=1)[0].strip())
    candidates.append(cleaned)

    unique_candidates = []
    for candidate in candidates:
        if candidate and candidate not in unique_candidates:
            unique_candidates.append(candidate)
    return unique_candidates


def request_vworld_geocode(session, address, api_key, address_type="road"):
    """VWorld Geocoder API에 주소 1건을 요청합니다."""
    params = {
        "service": "address",
        "request": "getcoord",
        "version": "2.0",
        "crs": "EPSG:4326",
        "address": address,
        "format": "json",
        "type": address_type,
        "key": api_key,
    }
    response = session.get(VWORLD_GEOCODE_URL, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def geocode_address_vworld(address, api_key, session):
    """주소 1건을 경도/위도로 변환합니다. 도로명주소 실패 시 지번주소 타입도 시도합니다."""
    last_status = None
    last_error = None

    for candidate in normalize_address_candidates(address):
        for address_type in ["road", "parcel"]:
            result_json = request_vworld_geocode(
                session=session,
                address=candidate,
                api_key=api_key,
                address_type=address_type,
            )
            response_body = result_json.get("response", {})
            status = response_body.get("status")
            last_status = status

            if status == "OK":
                point = response_body.get("result", {}).get("point", {})
                return {
                    "경도": float(point["x"]),
                    "위도": float(point["y"]),
                    "지오코딩주소": candidate,
                    "지오코딩유형": address_type,
                    "지오코딩상태": status,
                }

            error = response_body.get("error", {})
            error_code = error.get("code")
            error_text = error.get("text")
            last_error = error_text or status

            # 인증키 문제나 일일 요청량 초과는 뒤 주소를 계속 조회해도 해결되지 않으므로 즉시 중단합니다.
            if error_code in {"INVALID_KEY", "INCORRECT_KEY", "UNAVAILABLE_KEY", "OVER_REQUEST_LIMIT"}:
                raise RuntimeError(f"VWorld API 오류: {error_code} - {error_text}")

    return {
        "경도": pd.NA,
        "위도": pd.NA,
        "지오코딩주소": pd.NA,
        "지오코딩유형": pd.NA,
        "지오코딩상태": last_error or last_status or "NOT_FOUND",
    }

In [6]:
if not VWORLD_API_KEY:
    raise ValueError("VWORLD_API_KEY에 VWorld 인증키를 입력하거나 환경변수 VWORLD_API_KEY를 설정하십시오.")

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"입력 CSV를 찾지 못했습니다: {INPUT_CSV}")

facilities_df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

required_cols = ["시설ID", "기관소재지"]
missing_cols = [col for col in required_cols if col not in facilities_df.columns]
if missing_cols:
    raise ValueError(f"입력 CSV에 필요한 컬럼이 없습니다: {missing_cols}")

print(f"입력 시설 수: {len(facilities_df):,}")
display(facilities_df.head())

입력 시설 수: 487


,시설ID,시설유형,자치구,장기요양기관기호,기관명칭,전화,기관소재지
0,0,시트1 노인요양시설(241),종로구,11111000006,청운노인요양원,02-3217-0057,서울시 종로구 비봉길 76
1,1,시트1 노인요양시설(241),종로구,11111000056,인자인케어센터,02-394-2234,서울시 종로구 평창17길 26
2,2,시트1 노인요양시설(241),종로구,11111000060,평창동시니어센터,02-391-7936,서울시 종로구 평창15길 10
3,3,시트1 노인요양시설(241),종로구,11111000068,아름다운뜰안에요양원,02-396-0461,서울 종로구 평창21길 70
4,4,시트1 노인요양시설(241),종로구,11111000076,서울여자간호대학교 휴먼캐슬,02-391-8464,서울시 종로구 평창길 318


In [7]:
session = requests.Session()
geocoded_rows = []

for idx, row in facilities_df.iterrows():
    geocoded = geocode_address_vworld(
        address=row["기관소재지"],
        api_key=VWORLD_API_KEY,
        session=session,
    )
    geocoded_rows.append({**row.to_dict(), **geocoded})

    if (idx + 1) % 50 == 0 or (idx + 1) == len(facilities_df):
        print(f"진행: {idx + 1:,}/{len(facilities_df):,}")

    if SLEEP_SECONDS:
        time.sleep(SLEEP_SECONDS)

geocoded_df = pd.DataFrame(geocoded_rows)
geocoded_df["경도"] = pd.to_numeric(geocoded_df["경도"], errors="coerce")
geocoded_df["위도"] = pd.to_numeric(geocoded_df["위도"], errors="coerce")

success_count = geocoded_df[["경도", "위도"]].notna().all(axis=1).sum()
failed_count = len(geocoded_df) - success_count

print(f"좌표 변환 성공: {success_count:,}")
print(f"좌표 변환 실패: {failed_count:,}")
display(geocoded_df.head())

진행: 50/487
진행: 100/487
진행: 150/487
진행: 200/487
진행: 250/487
진행: 300/487
진행: 350/487
진행: 400/487
진행: 450/487
진행: 487/487
좌표 변환 성공: 453
좌표 변환 실패: 34


,시설ID,시설유형,자치구,장기요양기관기호,기관명칭,전화,기관소재지,경도,위도,지오코딩주소,지오코딩유형,지오코딩상태
0,0,시트1 노인요양시설(241),종로구,11111000006,청운노인요양원,02-3217-0057,서울시 종로구 비봉길 76,126.954960,37.615182,서울시 종로구 비봉길 76,road,OK
1,1,시트1 노인요양시설(241),종로구,11111000056,인자인케어센터,02-394-2234,서울시 종로구 평창17길 26,126.966664,37.612864,서울시 종로구 평창17길 26,road,OK
2,2,시트1 노인요양시설(241),종로구,11111000060,평창동시니어센터,02-391-7936,서울시 종로구 평창15길 10,126.965687,37.610613,서울시 종로구 평창15길 10,road,OK
3,3,시트1 노인요양시설(241),종로구,11111000068,아름다운뜰안에요양원,02-396-0461,서울 종로구 평창21길 70,126.968296,37.611537,서울 종로구 평창21길 70,road,OK
4,4,시트1 노인요양시설(241),종로구,11111000076,서울여자간호대학교 휴먼캐슬,02-391-8464,서울시 종로구 평창길 318,126.977217,37.614007,서울시 종로구 평창길 318,road,OK


In [8]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
geocoded_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"좌표 포함 CSV 저장 완료: {OUTPUT_CSV}")

failed_df = geocoded_df.loc[geocoded_df[["경도", "위도"]].isna().any(axis=1)].copy()
if not failed_df.empty:
    failed_df.to_csv(FAILED_CSV, index=False, encoding="utf-8-sig")
    print(f"좌표 변환 실패 목록 저장: {FAILED_CSV}")

display(geocoded_df[["시설ID", "기관명칭", "기관소재지", "경도", "위도", "지오코딩상태"]].head())

좌표 포함 CSV 저장 완료: Data/서울시_노인의료복지시설현황_geocoded.csv
좌표 변환 실패 목록 저장: Data/서울시_노인의료복지시설현황_geocoding_failed.csv


,시설ID,기관명칭,기관소재지,경도,위도,지오코딩상태
0,0,청운노인요양원,서울시 종로구 비봉길 76,126.954960,37.615182,OK
1,1,인자인케어센터,서울시 종로구 평창17길 26,126.966664,37.612864,OK
2,2,평창동시니어센터,서울시 종로구 평창15길 10,126.965687,37.610613,OK
3,3,아름다운뜰안에요양원,서울 종로구 평창21길 70,126.968296,37.611537,OK
4,4,서울여자간호대학교 휴먼캐슬,서울시 종로구 평창길 318,126.977217,37.614007,OK
